# MOMENT embeddings — DIMER task-inference tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook spec:** `1.0`  
**Capability:** pretrained time-series representation extraction  
**Upstream model:** `AutonLab/MOMENT-1-base` at immutable revision `9fea447e740eb968a9e8d80c7562ae122bdb5dde`

This notebook extracts pooled representations from the pretrained MOMENT encoder through the repository's public `moment_pipeline` API. **No gradient training, fine-tuning, in-context conditioning, or fitted preprocessing occurs.** Input validation/canonicalization is deterministic preprocessing only.

**Upstream vs. this repository.** Upstream MOMENT supplies the pretrained encoder and embedding operation. This repository supplies the immutable model pin, safetensors integrity checks, long-format input validation/canonicalization, missingness disclosures, output schema, and provenance export.

**By the end of this notebook you will be able to:**

- verify the runtime and immutable model identity;
- load a deterministic synthetic sample or an optional BYOD CSV;
- validate and canonicalize long-format time-series input;
- extract pooled MOMENT embeddings through the production-facing API;
- interpret embedding shape, pooling, channel, and missingness semantics;
- export identifier-preserving embeddings and provenance.

**This notebook does not demonstrate:** classification, forecasting, anomaly decisions, fine-tuning, or evidence that the embeddings are suitable for any specific downstream task. Embeddings are representations, not predictions.

References: [repository README](https://github.com/kurtvalcorza/moment-pipeline), [model card](https://github.com/kurtvalcorza/moment-pipeline/blob/main/MODEL_CARD.md), [sample dataset card](https://github.com/kurtvalcorza/moment-pipeline/blob/main/examples/sample-data/DATASET_CARD.md), [upstream MOMENT](https://github.com/moment-timeseries-foundation-model/moment), and [pinned model repository](https://huggingface.co/AutonLab/MOMENT-1-base).


## Prerequisites and data contract

- **Runtime:** Python 3.12; CPU is the default and no GPU is required. The public v1 API accepts `float32` only.
- **Network:** the first run needs access to GitHub and Hugging Face to install the repository and retrieve the pinned ~454 MB `model.safetensors`.
- **Credentials:** none are required for the default public model path.
- **Default data:** deterministic synthetic data generated by this repository. It is tutorial evidence, not benchmark or production-fitness evidence.
- **BYOD schema:** one UTF-8 CSV with columns `series_id`, `timestamp`, `channel`, `value`. `value` must be numeric or missing; identifiers and timestamps must be valid. Duplicate or ambiguous column names are rejected from the raw CSV header before dataframe parsing, and duplicate `(series_id, channel, timestamp)` rows are rejected by production validation.
- **Operational ceilings:** at most 5,000,000 rows, 1,024 series, 32 channels, and 1,024 canonical windows. MOMENT uses 512-step windows and 8-step non-overlapping patches. Short series are left-padded; long series keep the final 512 timestamps; irregular spacing is surfaced rather than silently interpolated.
- **BYOD privacy:** the pipeline does not send uploaded CSV contents to an external inference service. In Colab, the upload is stored in the hosted notebook runtime; model files are fetched separately from Hugging Face. Do not upload confidential, restricted, or sensitive data unless that hosted environment is authorized for it.

The default path is non-interactive. Set `USE_BYOD=True` only when you intentionally want an upload dialog.

Reproducibility note: the tutorial uses no stochastic sampling or random split. CPU inference runs in evaluation mode. Small numerical differences can still arise across different hardware, framework builds, or low-level kernels; this notebook does not claim bitwise determinism across platforms.


## 1. Bootstrap the repository and locked runtime

This stage installs the pinned `uv` bootstrap when needed, obtains the repository checkout, and installs the repository's exported lock graph before installing this package without resolving a second dependency graph. Successful completion means the repository checkout is identified and the locked runtime is installed without intentional dependency drift; it does **not** yet verify or execute the model weights.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/moment-pipeline.git"
REPO_NAME = "moment-pipeline"
UV_VERSION = "0.12.9"

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}"],
        check=True,
    )
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
    ROOT = Path.cwd()
    subprocess.run(
        ["uv", "pip", "install", "--system", "-r", "requirements.lock.txt"],
        check=True,
    )
    subprocess.run(
        ["uv", "pip", "install", "--system", "--no-deps", "-e", "."],
        check=True,
    )
else:
    print(f"Repository checkout detected: {ROOT}")

repo_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("repository commit:", repo_commit)


## 2. Inspect runtime, model identity, and limits

This stage exposes the effective Python/library versions, device and precision policy, immutable model identity, and resource ceilings **before inference**. Successful output means the notebook is running with a supported visible execution contract; it does not mean the model checkpoint has passed integrity verification yet.


In [ ]:
import platform
from importlib.metadata import version

import torch

from moment_pipeline import (
    MomentConfig,
    PINNED_MODEL_ID,
    PINNED_REVISION,
    __version__ as moment_pipeline_version,
)

TASK = "embedding"
config = MomentConfig(task=TASK, device="cpu")
limits = config.limits

print("Python:", platform.python_version())
print("moment-pipeline:", moment_pipeline_version)
print("momentfm:", version("momentfm"))
print("PyTorch:", torch.__version__)
print("device:", config.resolved_device())
print("dtype:", config.dtype)
print("model:", PINNED_MODEL_ID)
print("immutable revision:", PINNED_REVISION)
print(
    "limits:",
    {
        "max_rows": limits.max_rows,
        "max_series": limits.max_series,
        "max_channels": limits.max_channels,
        "max_windows": limits.max_windows,
        "sequence_length": config.sequence_length,
        "patch_length": config.patch_length,
    },
)


## 3. Load the deterministic sample or BYOD

The default sample is generated from repository code and verified against `SHA256SUMS`. The optional upload path first validates the **raw CSV header** with `read_long_csv_bytes()` so duplicate or ambiguous names cannot be silently renamed by pandas; the resulting frame still goes through the same production validation/canonicalization path in the next stage. Successful completion means one identified input frame is available and its source/digest are recorded.


In [ ]:
import hashlib
import json

import pandas as pd

from moment_pipeline import read_long_csv_bytes

USE_BYOD = False  # @param {type:"boolean"}

if USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "BYOD upload is available in Colab. Outside Colab, load a CSV into `frame` "
            "with columns series_id,timestamp,channel,value and validate it before inference."
        ) from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV with columns series_id,timestamp,channel,value.")
    name, payload = next(iter(uploaded.items()))
    frame = read_long_csv_bytes(payload)
    sample_identity = {
        "kind": "byod",
        "name": name,
        "sha256": hashlib.sha256(payload).hexdigest(),
    }
    print(f"Loaded BYOD: {name}")
else:
    sample_root = ROOT / "examples" / "sample-data"
    subprocess.run([sys.executable, str(sample_root / "generate_samples.py")], check=True)
    sample_path = sample_root / "moment_clean.csv"
    manifest = {}
    for line in (sample_root / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        digest, filename = line.split("  ", 1)
        manifest[filename] = digest
    observed = hashlib.sha256(sample_path.read_bytes()).hexdigest()
    assert observed == manifest[sample_path.name], "sample digest mismatch"
    frame = pd.read_csv(sample_path)
    sample_identity = {"kind": "synthetic", "name": sample_path.name, "sha256": observed}
    print(f"Loaded verified synthetic sample: {sample_path}")

print(frame.head())
print("sample identity:", sample_identity)


## 4. Validate and canonicalize

This stage runs the repository's production-facing schema/value/resource checks and converts the long frame to the exact 512-step representation consumed by MOMENT. Successful output means the input passed validation and any padding, truncation, source missingness, or irregular frequency is disclosed before model execution.


In [ ]:
from moment_pipeline import to_windows, validate_long_frame

report, normalized = validate_long_frame(frame, config)
windows = to_windows(normalized, config, report=report, frame=normalized)

print(
    "validation:",
    {
        "rows": report.n_rows,
        "series": len(report.series_ids),
        "channels": len(report.channels),
        "irregular_series": list(report.irregular_series),
    },
)
print("window tensor:", windows.x_enc.shape)
print("channels:", windows.channels)
print("padded windows:", int(sum(windows.padded)), "/", windows.n_windows)
print("truncated windows:", int(sum(windows.truncated)), "/", windows.n_windows)
print("source missing fraction:", windows.masked_point_fraction)
if any(windows.truncated):
    print("WARNING: long input series were truncated to their final 512 timestamps.")
if any(windows.padded):
    print("NOTE: short input series were left-padded; padding is excluded by the input mask.")


## 5. Resolve the pinned checkpoint and extract embeddings

`load_moment()` verifies the immutable model snapshot and loads only safetensors-backed state; `embed()` then exercises the repository's supported embedding API. Successful execution proves that this validated input can be processed by the verified pinned encoder and exposes the effective embedding contract; it does not establish downstream task quality.


In [ ]:
from moment_pipeline import build_provenance, embed, load_moment

model = load_moment(task="embedding", device="cpu")
result = embed(windows, model, warmup=False)
provenance = build_provenance(model, windows, result)

print("effective model:", model.identity.name)
print("effective revision:", model.identity.revision)
print("verified weight file:", model.identity.weight_file_loaded)
print("embedding shape:", result.embeddings.shape)
print("reduction:", result.reduction)
print("channel policy:", result.channel_policy)
print("missingness visible to model:", result.missingness_visible_to_model)


## 6. Interpret and export embeddings

There is **no intrinsic accuracy metric for an embedding vector**. Representation quality must be evaluated on a labelled or otherwise externally checkable downstream task appropriate to the use case—for example linear-probe classification, labelled retrieval precision, or a justified clustering validation design. The L2 norm below is only a finiteness/sanity check, not a quality score.

MOMENT's upstream `embed` path has no per-point observedness mask. Pre-filled missing positions are visible to the encoder, so missing-data fractions must be interpreted alongside the vectors.

Successful completion of this stage writes `outputs/moment_embeddings.csv` with input identifiers alongside vectors and `outputs/moment_embeddings_provenance.json` with model/runtime/data provenance. File creation establishes a machine-readable handoff boundary; it does **not** establish that the vectors are useful for a downstream task.


In [ ]:
output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

embedding_frame = result.to_frame()
embedding_frame.to_csv(output_dir / "moment_embeddings.csv", index=False)
provenance["data"] = sample_identity
(output_dir / "moment_embeddings_provenance.json").write_text(
    json.dumps(provenance, indent=2, default=str),
    encoding="utf-8",
)

print(embedding_frame.iloc[:, :8])
print("embedding L2 norm (sanity only):", float((result.embeddings[0] ** 2).sum() ** 0.5))
print("exports:", sorted(path.name for path in output_dir.glob("moment_embeddings*")))


## Interpretation, limits, and next steps

A successful run proves that the repository can validate this input, resolve and integrity-check the pinned MOMENT checkpoint, execute the supported pooled-embedding path, and export identifier-preserving vectors with provenance.

It **does not prove** that these embeddings are accurate for classification, retrieval, clustering, forecasting, or any production domain; it also does not prove that missing values were ignored by the encoder. Validate representation usefulness on a downstream task with domain-appropriate labelled evidence before deployment.

Useful next experiments are to compare downstream linear-probe or retrieval performance on clean versus missingness-bearing windows, or to evaluate task-specific representations on an independent dataset.
